In [2]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/terriljoel/retrieval-grounded-remote-sensing.git"
REPO_REF = "feat/object-detection-YOLO"  # Change to main after merging.
REPO_DIR = Path("/content/retrieval-grounded-remote-sensing")
SHARED_ROOT = Path("/content/drive/Othercomputers/My laptop/shared_resources")

drive.mount("/content/drive")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Repository path exists but is not a Git clone: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)],
    check=True,
)

if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared resources were not found at {SHARED_ROOT}. "
        "Check the Google Drive computer and folder names."
    )

os.environ.update({
    "SHARED_RESOURCES_ROOT": str(SHARED_ROOT),
    "RAW_DATASET_ROOT": "/content/datasets/raw",
    "PROCESSED_DATASET_ROOT": "/content/datasets/processed",
    "MANIFEST_ROOT": str(SHARED_ROOT / "datasets" / "manifests"),
    "EXPERIMENT_OUTPUT_ROOT": str(SHARED_ROOT / "experiment_outputs"),
})

Path(os.environ["MANIFEST_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["EXPERIMENT_OUTPUT_ROOT"]).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

print(f"Ready. Working directory: {Path.cwd()}")
print("Next command:")
print("!rs-prepare-detector --config configs/detection/yolov8n_baseline.yaml")

Mounted at /content/drive
Ready. Working directory: /content/retrieval-grounded-remote-sensing
Next command:
!rs-prepare-detector --config configs/detection/yolov8n_baseline.yaml


In [2]:
os.environ["SHARED_RESOURCES_ROOT"]

'/content/drive/Othercomputers/My laptop/shared_resources'

In [6]:
!tail -n 20 configs/inference/ultralytics_export.yaml

# available, use format: yolo or format: nwpu and point path to the label folder.
ground_truth: 
  format: nwpu
  path: "${SHARED_RESOURCES_ROOT}/datasets/raw/NWPU VHR-10 dataset/ground truth"
  iou_threshold: 0.5

inference:
  imgsz: 1024
  batch: 16
  confidence: 0.05
  iou: 0.70
  device: 0
  save_rendered_images: true
  save_yolo_labels: true
  save_confidence: true
  verbose: true
  exist_ok: false

outputs:
  root: "${SHARED_RESOURCES_ROOT}/inference"


In [7]:
!rs-export-predictions --config configs/inference/ultralytics_export.yaml

[job] id=20260920T152952Z_5969666e
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-export-predictions/20260920T152952Z_5969666e

0: 1024x1024 (no detections), 18.1ms
1: 1024x1024 (no detections), 18.1ms
2: 1024x1024 (no detections), 18.1ms
3: 1024x1024 (no detections), 18.1ms
4: 1024x1024 (no detections), 18.1ms
5: 1024x1024 (no detections), 18.1ms
6: 1024x1024 1 baseball_diamond, 1 ground_track_field, 18.1ms
7: 1024x1024 (no detections), 18.1ms
8: 1024x1024 (no detections), 18.1ms
9: 1024x1024 1 airplane, 18.1ms
10: 1024x1024 (no detections), 18.1ms
11: 1024x1024 (no detections), 18.1ms
12: 1024x1024 1 basketball_court, 1 ground_track_field, 18.1ms
13: 1024x1024 (no detections), 18.1ms
14: 1024x1024 (no detections), 18.1ms
15: 1024x1024 (no detections), 18.1ms
Speed: 9.2ms preprocess, 18.1ms inference, 1.4ms postprocess per image at shape (16, 3, 1024, 1024)
Results saved to /content/drive/Othercomputers/My laptop/shared_resource

In [8]:
import os
from pathlib import Path

import pandas as pd

inference_root = Path(os.environ["SHARED_RESOURCES_ROOT"]) / "inference"

exports = sorted(
    [
        path
        for path in inference_root.iterdir()
        if path.is_dir() and (path / "inference_metadata.json").is_file()
    ],
    key=lambda path: path.stat().st_mtime,
)

if not exports:
    raise FileNotFoundError(f"No complete inference exports under {inference_root}")

export_dir = exports[-1]
print("Selected export:", export_dir)

required_files = [
    "inference_images.csv",
    "detections.csv",
    "ground_truth.csv",
    "prediction_comparison.csv",
    "inference_metadata.json",
    "resolved_config.yaml",
]

for filename in required_files:
    path = export_dir / filename
    print(f"{filename}: {'OK' if path.is_file() else 'MISSING'}")
    assert path.is_file(), path

images = pd.read_csv(export_dir / "inference_images.csv")
detections = pd.read_csv(export_dir / "detections.csv")
ground_truth = pd.read_csv(export_dir / "ground_truth.csv")
comparisons = pd.read_csv(export_dir / "prediction_comparison.csv")

print("\nCounts")
print("Images:", len(images))
print("Detections:", len(detections))
print("Ground-truth objects:", len(ground_truth))
print("Comparison records:", len(comparisons))

print("\nImages by split")
display(
    images.groupby("source_split")
    .size()
    .rename("images")
    .reset_index()
)

comparison_with_split = comparisons.merge(
    images[["image_id", "source_split"]],
    on="image_id",
    how="left",
    validate="many_to_one",
)

print("\nComparison status by split")
display(
    comparison_with_split.groupby(["source_split", "status"])
    .size()
    .rename("records")
    .reset_index()
)

assert len(images) == 800
assert images["image_id"].is_unique
assert images["source_image_path"].nunique() == 800
assert images["source_split"].value_counts().to_dict() == {
    "train": 562,
    "val": 120,
    "test": 118,
}
assert comparison_with_split["source_split"].notna().all()

print("\nExport integrity checks passed.")

Selected export: /content/drive/Othercomputers/My laptop/shared_resources/inference/nwpu_yolov8s_1024_final_predictions_nwpu_vhr10_manifest-3
inference_images.csv: OK
detections.csv: OK
ground_truth.csv: OK
prediction_comparison.csv: OK
inference_metadata.json: OK
resolved_config.yaml: OK

Counts
Images: 800
Detections: 4869
Ground-truth objects: 3896
Comparison records: 4892

Images by split


,source_split,images
0,test,118
1,train,562
2,val,120



Comparison status by split


,source_split,status,records
0,test,class_error,2
1,test,false_negative,14
2,test,false_positive,148
3,test,true_positive,507
4,train,class_error,21
5,train,false_positive,702
6,train,true_positive,2821
7,val,false_negative,9
8,val,false_positive,146
9,val,true_positive,522



Export integrity checks passed.


In [9]:
import numpy as np
import pandas as pd

# Keep only real detector proposals from validation.
# False negatives have no detector proposal and are evaluated separately.
val_cases = (
    comparison_with_split[
        (comparison_with_split["source_split"] == "val")
        & comparison_with_split["status"].isin(
            ["true_positive", "false_positive", "class_error"]
        )
        & comparison_with_split["detection_id"].notna()
    ]
    .merge(
        detections[["detection_id", "confidence"]],
        on="detection_id",
        how="left",
        validate="one_to_one",
    )
    .copy()
)

val_cases["correct"] = val_cases["status"] == "true_positive"
val_cases = val_cases.sort_values(
    ["confidence", "detection_id"],
    ascending=[False, True],
).reset_index(drop=True)

# Risk among all proposals accepted at each confidence threshold.
val_cases["accepted"] = np.arange(1, len(val_cases) + 1)
val_cases["errors"] = (~val_cases["correct"]).cumsum()
val_cases["coverage"] = val_cases["accepted"] / len(val_cases)
val_cases["risk"] = val_cases["errors"] / val_cases["accepted"]

coverage = np.concatenate([[0.0], val_cases["coverage"].to_numpy()])
risk = np.concatenate([[0.0], val_cases["risk"].to_numpy()])
aurc = np.trapezoid(risk, coverage)

def operating_point_at_budget(table, budget):
    eligible = table[table["risk"] <= budget]

    if eligible.empty:
        return {
            "budget": budget,
            "coverage": 0.0,
            "threshold": None,
            "accepted": 0,
            "errors": 0,
        }

    row = eligible.iloc[-1]

    return {
        "budget": budget,
        "coverage": float(row["coverage"]),
        "threshold": float(row["confidence"]),
        "accepted": int(row["accepted"]),
        "errors": int(row["errors"]),
    }

operating_points = pd.DataFrame(
    [
        operating_point_at_budget(val_cases, 0.01),
        operating_point_at_budget(val_cases, 0.05),
    ]
)

print("Validation proposals:", len(val_cases))
print(val_cases["status"].value_counts())
print(f"Detector-confidence AURC: {aurc:.4f}")

display(operating_points)

Validation proposals: 668
status
true_positive     522
false_positive    146
Name: count, dtype: int64
Detector-confidence AURC: 0.0440


,budget,coverage,threshold,accepted,errors
0,0.01,0.088323,0.902551,59,0
1,0.05,0.784431,0.611395,524,26


In [10]:
baseline_output = export_dir / "validation_detector_risk_coverage.csv"
val_cases.to_csv(baseline_output, index=False)

operating_output = export_dir / "validation_detector_operating_points.csv"
operating_points.to_csv(operating_output, index=False)

print("Saved:", baseline_output)
print("Saved:", operating_output)

Saved: /content/drive/Othercomputers/My laptop/shared_resources/inference/nwpu_yolov8s_1024_final_predictions_nwpu_vhr10_manifest-3/validation_detector_risk_coverage.csv
Saved: /content/drive/Othercomputers/My laptop/shared_resources/inference/nwpu_yolov8s_1024_final_predictions_nwpu_vhr10_manifest-3/validation_detector_operating_points.csv


# Convert HRRSD dataset

In [3]:
import os
from pathlib import Path

shared_root = Path(os.environ["SHARED_RESOURCES_ROOT"])
hrrsd_root = shared_root / "datasets/raw/HRRSD_external_subset_seed42"

print("Dataset:", hrrsd_root)
print("Images:", (hrrsd_root / "JPEGImages").is_dir())
print("Annotations:", (hrrsd_root / "Annotations").is_dir())

Dataset: /content/drive/Othercomputers/My laptop/shared_resources/datasets/raw/HRRSD_external_subset_seed42
Images: True
Annotations: True


In [ ]:
!rs-convert-dataset --config configs/datasets/hrrsd_external.yaml

[job] id=20260922T001554Z_42b54d4f
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-convert-dataset/20260922T001554Z_42b54d4f
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (94434015 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


## Export predictions

In [ ]:
%cd /content/retrieval-grounded-remote-sensing

!git pull --ff-only
!pip install -e .

!rs-export-predictions --config configs/inference/hrrsd_yolov8s_1024_export.yaml